In [2]:
import pandas as pd
import numpy as np

# 1. Load with header=None so row 0 is not swallowed as column names
DATA_PATH = "610_ps4_training/trainingT2HAR/har_train.csv"  # <-- Update filename
df = pd.read_csv(DATA_PATH, header=None)

print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("\nData Types Summary:")
print(df.dtypes.value_counts())

# 2. Check the first and last 3 columns to locate the target label (and optional subject ID)
print("\nFirst 3 columns (head):")
display(df.iloc[:5, :3])

print("\nLast 3 columns (head):")
display(df.iloc[:5, -3:])

# 3. Check unique values in candidate label columns (first and last column)
for idx in [0, df.shape[1] - 1]:
    n_unique = df.iloc[:, idx].nunique()
    print(f"\nColumn {idx} -> {n_unique} unique values: {df.iloc[:, idx].unique()[:10]}")

Dataset Shape: 2,442 rows x 329 columns

Data Types Summary:
float64    328
str          1
Name: count, dtype: int64

First 3 columns (head):


,0,1,2
0,0.0,50.140426,0.059579
1,0.0,39.458743,0.598236
2,0.0,273.775013,0.306231
3,0.0,-0.124842,0.058591
4,0.0,50.004188,0.117917



Last 3 columns (head):


,326,327,328
0,0.285714,0.238095,Tennis
1,0.285714,0.238095,Sitting
2,0.285714,0.238095,Cycling
3,0.285714,0.238095,Tennis
4,0.285714,0.238095,Football



Column 0 -> 3 unique values: [0.         0.95238095 1.9047619 ]

Column 328 -> 11 unique values: <ArrowStringArray>
[           'Tennis',           'Sitting',           'Cycling',
          'Football',           'Walking',          'Swimming',
   'WalkingUpstairs', 'WalkingDownstairs',          'JumpRope',
           'Jogging']
Length: 10, dtype: str


In [3]:
# Adjust target_col_idx based on Step 1 output (usually df.shape[1] - 1 or 0)
target_col_idx = df.shape[1] - 1

# Rename features as feat_0, feat_1, ... and label as 'activity'
feature_indices = [i for i in range(df.shape[1]) if i != target_col_idx]
rename_map = {col_idx: f"feat_{i}" for i, col_idx in enumerate(feature_indices)}
rename_map[target_col_idx] = "activity"

df = df.rename(columns=rename_map)

# If 'activity' contains string labels (e.g., 'WALKING', 'SITTING'), encode to integers
if df["activity"].dtype == "object":
    df["activity_name"] = df["activity"]
    df["activity"] = df["activity_name"].astype("category").cat.codes
    label_mapping = dict(enumerate(df["activity_name"].astype("category").cat.categories))
    print("Encoded Activity Mapping:", label_mapping)

print("\nClean Columns Preview:", list(df.columns[:5]), "...", list(df.columns[-3:]))


Clean Columns Preview: ['feat_0', 'feat_1', 'feat_2', 'feat_3', 'feat_4'] ... ['feat_326', 'feat_327', 'activity']


In [4]:
df.head()

,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,...,feat_319,feat_320,feat_321,feat_322,feat_323,feat_324,feat_325,feat_326,feat_327,activity
0,0.0,50.140426,0.059579,0.0,0.492944,9.664197e-03,0.149931,0.000000,35.196034,3.268344,...,0.333333,0.047619,0.095238,0.0,0.0,0.380952,0.333333,0.285714,0.238095,Tennis
1,0.0,39.458743,0.598236,0.0,0.537163,2.711785e-04,0.000274,-0.660429,143.137411,7.839429,...,0.333333,0.047619,0.095238,0.0,0.0,0.380952,0.333333,0.285714,0.238095,Sitting
2,0.0,273.775013,0.306231,0.0,0.500422,7.866959e-01,0.011083,-217.358708,143.205373,59.754427,...,0.333333,0.047619,0.095238,0.0,0.0,0.380952,0.333333,0.285714,0.238095,Cycling
3,0.0,-0.124842,0.058591,0.0,0.500000,8.599624e-36,0.045089,-8.708700,0.281777,0.445135,...,0.333333,0.047619,0.095238,0.0,0.0,0.380952,0.333333,0.285714,0.238095,Tennis
4,0.0,50.004188,0.117917,0.0,0.469343,5.580454e-01,0.076807,-55.242511,44.912627,10.519441,...,0.333333,0.047619,0.095238,0.0,0.0,0.380952,0.333333,0.285714,0.238095,Football


In [5]:
import sys

!{sys.executable} -m pip install --user umap-learn

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 2.8/2.8 MB 33.1 MB/s  0:00:00
   ---------------------------------------- 0.0/43.0 MB ? eta -:--:--
   --------- ------------------------------ 10.2/43.0 MB 50.4 MB/s eta 0:00:01
   ------------------ --------------------- 19.9/43.0 MB 48.7 MB/s eta 0:00:01
   ------------------------------ --------- 32.2/43.0 MB 52.6 MB/s eta 0:00:01
   ---------------------------------------  42.2/43.0 MB 52.3 MB/s eta 0:00:01
   ---------------------------------------- 43.0/43.0 MB 48.7 MB/s  0:00:00

   ---------------------------------------- 0/4 [llvmlite]
   ---------------------------------------- 0/4 [llvmlite]
   ---------------------------------------- 0/4 [llvmlite]
   ---------- ----------------------------- 1/4 [numba]
   ---------- ----------------------------- 1/4 [numba]
   ---------- ----------------------------- 1/4 [numba]
   ---------- ----------------------------- 1/4 [

In [7]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "4"

import random
import time
import warnings
import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
import umap
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---------------------------------------------------------
# GLOBAL RANDOM SEED FOR REPRODUCIBILITY
# ---------------------------------------------------------
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# =========================================================
# STAGE 1: LOAD UNNAMED DATASET & STRATIFIED SPLIT
# =========================================================
DATA_PATH = "610_ps4_training/trainingT2HAR/har_train.csv"  # <-- Update to your exact HAR CSV path
df = pd.read_csv(DATA_PATH, header=None)

target_idx = df.shape[1] - 1 if df.iloc[:, -1].nunique() <= 30 else 0
feat_indices = [i for i in range(df.shape[1]) if i != target_idx]
rename_map = {col_idx: f"feat_{i}" for i, col_idx in enumerate(feat_indices)}
rename_map[target_idx] = "activity"
df = df.rename(columns=rename_map)

le = LabelEncoder()
y_all = le.fit_transform(df["activity"].astype(str))
class_names = [str(c) for c in le.classes_]

feature_cols = [c for c in df.columns if c != "activity"]
X_all = df[feature_cols].apply(pd.to_numeric, errors="coerce").values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=RANDOM_SEED
)

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
X_train = scaler.fit_transform(imputer.fit_transform(X_train_raw))
X_test = scaler.transform(imputer.transform(X_test_raw))

print("=" * 68)
print(f"STAGE 1 READY | Train: {X_train.shape} | Test: {X_test.shape} | Classes: {len(class_names)}")
print("=" * 68)

# =========================================================
# STAGE 2: FAST UMAP OPTIMAL FEATURE COUNT SEARCH
# =========================================================
# Warm up Numba JIT once so UMAP runs at full speed
_ = umap.UMAP(n_components=2, n_neighbors=5, random_state=RANDOM_SEED).fit_transform(
    X_train[:50]
)

umap_candidate_dims = [5, 10, 15, 25, 40]
cv_umap = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
umap_records = []

print("\nSTAGE 2: SEARCHING OPTIMAL UMAP FEATURE COUNT...")
for n_comp in umap_candidate_dims:
    t0 = time.time()
    fold_f1s = []
    for tr_idx, va_idx in cv_umap.split(X_train, y_train):
        reducer = umap.UMAP(
            n_components=n_comp,
            n_neighbors=15,
            min_dist=0.1,
            metric="euclidean",
            random_state=RANDOM_SEED,
        )
        Z_tr = reducer.fit_transform(X_train[tr_idx])
        Z_va = reducer.transform(X_train[va_idx])

        proxy_clf = RandomForestClassifier(
            n_estimators=150,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=4,
        )
        proxy_clf.fit(Z_tr, y_train[tr_idx])
        preds = proxy_clf.predict(Z_va)
        fold_f1s.append(f1_score(y_train[va_idx], preds, average="macro"))

    mean_f1 = float(np.mean(fold_f1s))
    elapsed = time.time() - t0
    umap_records.append(
        {"Method": "UMAP", "Num_Features": n_comp, "CV_Macro_F1": mean_f1}
    )
    print(f"  UMAP (n_components={n_comp:2d}) -> CV Macro F1: {mean_f1:.4f} ({elapsed:.1f}s)")

umap_df = pd.DataFrame(umap_records).sort_values("CV_Macro_F1", ascending=False)
print("\nUMAP FEATURE COUNT LEADERBOARD:")
print(umap_df.to_string(index=False))

BEST_N_FEATURES = int(umap_df.iloc[0]["Num_Features"])
print(
    f"\n>>> SELECTED OPTIMAL UMAP FEATURES: {BEST_N_FEATURES} "
    f"(CV Macro F1 = {umap_df.iloc[0]['CV_Macro_F1']:.4f})"
)

# Fit final UMAP on full X_train and transform X_test cleanly
best_umap = umap.UMAP(
    n_components=BEST_N_FEATURES,
    n_neighbors=15,
    min_dist=0.1,
    metric="euclidean",
    random_state=RANDOM_SEED,
)
Z_train = best_umap.fit_transform(X_train)
Z_test = best_umap.transform(X_test)

# =========================================================
# STAGE 3: AUTOML MODEL SEARCH ON OPTIMAL UMAP FEATURES
# =========================================================
def build_automl_model(trial):
    family = trial.suggest_categorical(
        "model_family",
        ["XGBoost", "LightGBM", "HistGradientBoosting", "RandomForest"],
    )

    if family == "XGBoost":
        return XGBClassifier(
            n_estimators=trial.suggest_int("xgb_n_estimators", 120, 350),
            learning_rate=trial.suggest_float("xgb_lr", 0.03, 0.2, log=True),
            max_depth=trial.suggest_int("xgb_max_depth", 3, 8),
            subsample=trial.suggest_float("xgb_subsample", 0.7, 1.0),
            colsample_bytree=trial.suggest_float("xgb_colsample", 0.7, 1.0),
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=4,
        )
    elif family == "LightGBM":
        return lgb.LGBMClassifier(
            n_estimators=trial.suggest_int("lgb_n_estimators", 120, 350),
            learning_rate=trial.suggest_float("lgb_lr", 0.03, 0.2, log=True),
            num_leaves=trial.suggest_int("lgb_num_leaves", 15, 63),
            subsample=trial.suggest_float("lgb_subsample", 0.7, 1.0),
            colsample_bytree=trial.suggest_float("lgb_colsample", 0.7, 1.0),
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=4,
            verbose=-1,
        )
    elif family == "HistGradientBoosting":
        return HistGradientBoostingClassifier(
            max_iter=trial.suggest_int("hgb_max_iter", 120, 350),
            learning_rate=trial.suggest_float("hgb_lr", 0.03, 0.2, log=True),
            max_leaf_nodes=trial.suggest_int("hgb_max_leaves", 15, 63),
            class_weight="balanced",
            random_state=RANDOM_SEED,
        )
    else:
        return RandomForestClassifier(
            n_estimators=trial.suggest_int("rf_n_estimators", 150, 400),
            max_depth=trial.suggest_int("rf_max_depth", 8, 25),
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=4,
        )


cv_automl = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)


def automl_objective(trial):
    scores = []
    for tr_idx, va_idx in cv_automl.split(Z_train, y_train):
        clf = build_automl_model(trial)
        clf.fit(Z_train[tr_idx], y_train[tr_idx])
        preds = clf.predict(Z_train[va_idx])
        scores.append(f1_score(y_train[va_idx], preds, average="macro"))
    return float(np.mean(scores))


print(f"\nSTAGE 3: RUNNING AUTOML ON UMAP-{BEST_N_FEATURES}...")
study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED)
)
study.optimize(automl_objective, n_trials=15)

trials_df = study.trials_dataframe()
automl_leaderboard = (
    trials_df[["number", "params_model_family", "value"]]
    .rename(
        columns={
            "number": "Trial",
            "params_model_family": "Model_Family",
            "value": "CV_Macro_F1",
        }
    )
    .sort_values("CV_Macro_F1", ascending=False)
)
print("\nTOP AUTOML MODELS (3-FOLD CV LEADERBOARD):")
print(automl_leaderboard.head(8).to_string(index=False))

best_trial = study.best_trial
best_family = best_trial.params["model_family"]
best_model = build_automl_model(best_trial)
best_model.fit(Z_train, y_train)

# =========================================================
# STAGE 4: FINAL PERFORMANCE METRICS ON HELD-OUT TEST SET
# =========================================================
y_pred_test = best_model.predict(Z_test)

acc = accuracy_score(y_test, y_pred_test)
macro_f1 = f1_score(y_test, y_pred_test, average="macro")
weighted_f1 = f1_score(y_test, y_pred_test, average="weighted")

print("\n" + "=" * 68)
print(f"STAGE 4: FINAL HELD-OUT TEST METRICS (UMAP-{BEST_N_FEATURES} + {best_family})")
print("=" * 68)
print(f"Dimensionality Reduction : UMAP ({BEST_N_FEATURES} features, down from {X_train.shape[1]})")
print(f"Best AutoML Model        : {best_family} (Trial {best_trial.number})")
print(f"Test Accuracy            : {acc * 100:.4f}%")
print(f"Test Macro F1            : {macro_f1:.4f}")
print(f"Test Weighted F1         : {weighted_f1:.4f}")
print("-" * 68)
print(classification_report(y_test, y_pred_test, target_names=class_names, digits=4))

STAGE 1 READY | Train: (1953, 328) | Test: (489, 328) | Classes: 11

STAGE 2: SEARCHING OPTIMAL UMAP FEATURE COUNT...
  UMAP (n_components= 5) -> CV Macro F1: 0.3827 (27.3s)
  UMAP (n_components=10) -> CV Macro F1: 0.3934 (30.4s)
  UMAP (n_components=15) -> CV Macro F1: 0.3890 (33.3s)
  UMAP (n_components=25) -> CV Macro F1: 0.4033 (36.2s)
  UMAP (n_components=40) -> CV Macro F1: 0.4142 (41.1s)

UMAP FEATURE COUNT LEADERBOARD:
Method  Num_Features  CV_Macro_F1
  UMAP            40     0.414174
  UMAP            25     0.403273
  UMAP            10     0.393434
  UMAP            15     0.388984
  UMAP             5     0.382732

>>> SELECTED OPTIMAL UMAP FEATURES: 40 (CV Macro F1 = 0.4142)

STAGE 3: RUNNING AUTOML ON UMAP-40...

TOP AUTOML MODELS (3-FOLD CV LEADERBOARD):
 Trial         Model_Family  CV_Macro_F1
     1 HistGradientBoosting     0.472146
    13         RandomForest     0.469041
    11         RandomForest     0.468760
     8         RandomForest     0.466426
     3        

In [9]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "4"

import random
import warnings
import numpy as np
import pandas as pd
import umap
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler, StandardScaler

warnings.filterwarnings("ignore")

# ---------------------------------------------------------
# 1. REPRODUCIBILITY & LOAD UNNAMED HAR DATASET
# ---------------------------------------------------------
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATA_PATH = "610_ps4_training/trainingT2HAR/har_train.csv"  # <-- Update to your exact HAR CSV path
df = pd.read_csv(DATA_PATH, header=None)

target_idx = df.shape[1] - 1 if df.iloc[:, -1].nunique() <= 30 else 0
feat_indices = [i for i in range(df.shape[1]) if i != target_idx]
rename_map = {col_idx: f"feat_{i}" for i, col_idx in enumerate(feat_indices)}
rename_map[target_idx] = "activity"
df = df.rename(columns=rename_map)

le = LabelEncoder()
y_all = le.fit_transform(df["activity"].astype(str))
class_names = [str(c) for c in le.classes_]

feature_cols = [c for c in df.columns if c != "activity"]
X_all = df[feature_cols].apply(pd.to_numeric, errors="coerce").values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=RANDOM_SEED
)

# ---------------------------------------------------------
# 2A. CURRENT SETUP: UNSUPERVISED UMAP-40 + HISTGRADIENTBOOSTING
# ---------------------------------------------------------
imp_std = SimpleImputer(strategy="median")
scl_std = StandardScaler()
X_tr_std = scl_std.fit_transform(imp_std.fit_transform(X_train_raw))
X_te_std = scl_std.transform(imp_std.transform(X_test_raw))

umap_base = umap.UMAP(
    n_components=40,
    n_neighbors=15,
    min_dist=0.1,
    metric="euclidean",
    random_state=RANDOM_SEED,
)
Z_tr_base = umap_base.fit_transform(X_tr_std)
Z_te_base = umap_base.transform(X_te_std)

hgb_base = HistGradientBoostingClassifier(
    max_iter=250,
    learning_rate=0.06,
    max_leaf_nodes=31,
    class_weight="balanced",
    random_state=RANDOM_SEED,
)
hgb_base.fit(Z_tr_base, y_train)
pred_base = hgb_base.predict(Z_te_base)

# ---------------------------------------------------------
# 2B. IMPROVED SETUP: SUPERVISED COSINE UMAP-40 + TUNED HGB
# ---------------------------------------------------------
# RobustScaler prevents extreme accelerometer/gyroscope outliers from skewing distances
imp_rob = SimpleImputer(strategy="median")
scl_rob = RobustScaler()
X_tr_rob = scl_rob.fit_transform(imp_rob.fit_transform(X_train_raw))
X_te_rob = scl_rob.transform(imp_rob.transform(X_test_raw))

# Supervised UMAP uses y_train ONLY during fit_transform to separate subtle classes
# (e.g. Walking vs. WalkingUpstairs vs. WalkingDownstairs)
umap_improved = umap.UMAP(
    n_components=40,
    n_neighbors=20,
    min_dist=0.05,
    metric="cosine",
    target_weight=0.5,
    random_state=RANDOM_SEED,
)
Z_tr_imp = umap_improved.fit_transform(X_tr_rob, y=y_train)
Z_te_imp = umap_improved.transform(X_te_rob)

hgb_improved = HistGradientBoostingClassifier(
    max_iter=350,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=10,
    l2_regularization=1.0,
    class_weight="balanced",
    random_state=RANDOM_SEED,
)
hgb_improved.fit(Z_tr_imp, y_train)
pred_imp = hgb_improved.predict(Z_te_imp)

# ---------------------------------------------------------
# 3. SIDE-BY-SIDE COMPARISON & FULL REPORT
# ---------------------------------------------------------
summary = []
for name, preds in [
    ("1. Standard Unsupervised UMAP-40 + HGB", pred_base),
    ("2. Improved Supervised Cosine UMAP-40 + HGB", pred_imp),
]:
    summary.append({
        "Pipeline": name,
        "Test Accuracy (%)": round(accuracy_score(y_test, preds) * 100, 4),
        "Macro F1": round(f1_score(y_test, preds, average="macro"), 4),
        "Weighted F1": round(f1_score(y_test, preds, average="weighted"), 4),
    })

print("=" * 72)
print("UMAP-40 + HISTGRADIENTBOOSTING: BEFORE vs. AFTER IMPROVEMENT")
print("=" * 72)
print(pd.DataFrame(summary).to_string(index=False))

print("\n" + "=" * 72)
print("DETAILED CLASSIFICATION REPORT (IMPROVED UMAP-40 + HGB)")
print("=" * 72)
print(classification_report(y_test, pred_imp, target_names=class_names, digits=4))

UMAP-40 + HISTGRADIENTBOOSTING: BEFORE vs. AFTER IMPROVEMENT
                                   Pipeline  Test Accuracy (%)  Macro F1  Weighted F1
     1. Standard Unsupervised UMAP-40 + HGB            62.9857    0.4168       0.6069
2. Improved Supervised Cosine UMAP-40 + HGB            49.8978    0.3237       0.5249

DETAILED CLASSIFICATION REPORT (IMPROVED UMAP-40 + HGB)
                   precision    recall  f1-score   support

          Cycling     0.7091    0.7091    0.7091        55
         Football     0.6400    0.4103    0.5000        78
          Jogging     0.0357    0.5000    0.0667         2
         JumpRope     0.1618    0.4783    0.2418        23
          Pushups     0.0000    0.0000    0.0000         3
          Sitting     0.8136    0.6906    0.7471       139
         Swimming     0.2857    0.4000    0.3333        25
           Tennis     0.5106    0.4138    0.4571       116
          Walking     0.3333    0.1667    0.2222        12
WalkingDownstairs     0.1333    0

In [10]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "4"

import random
import warnings
import numpy as np
import pandas as pd
import umap
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# 1. Load & Split
DATA_PATH = "610_ps4_training/trainingT2HAR/har_train.csv"  # <-- Update to your exact HAR CSV path
df = pd.read_csv(DATA_PATH, header=None)

target_idx = df.shape[1] - 1 if df.iloc[:, -1].nunique() <= 30 else 0
feat_indices = [i for i in range(df.shape[1]) if i != target_idx]
rename_map = {col_idx: f"feat_{i}" for i, col_idx in enumerate(feat_indices)}
rename_map[target_idx] = "activity"
df = df.rename(columns=rename_map)

le = LabelEncoder()
y_all = le.fit_transform(df["activity"].astype(str))
class_names = [str(c) for c in le.classes_]

feature_cols = [c for c in df.columns if c != "activity"]
X_all = df[feature_cols].apply(pd.to_numeric, errors="coerce").values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=RANDOM_SEED
)

imp = SimpleImputer(strategy="median")
scl = StandardScaler()
X_train = scl.fit_transform(imp.fit_transform(X_train_raw))
X_test = scl.transform(imp.transform(X_test_raw))

hgb_cfg = dict(
    max_iter=350,
    learning_rate=0.06,
    max_leaf_nodes=31,
    min_samples_leaf=10,
    l2_regularization=1.0,
    class_weight="balanced",
    random_state=RANDOM_SEED,
)

pipelines = {}

# A. Current UMAP-40 Baseline
reducer_umap = umap.UMAP(
    n_components=40, n_neighbors=15, min_dist=0.1, random_state=RANDOM_SEED
)
pipelines["1. UMAP-40 + HGB"] = (
    reducer_umap.fit_transform(X_train),
    reducer_umap.transform(X_test),
)

# B. SelectKBest (Top 80 Mutual Information features — preserves exact sensor values)
selector = SelectKBest(
    score_func=lambda X, y: mutual_info_classif(X, y, random_state=RANDOM_SEED),
    k=min(80, X_train.shape[1]),
)
pipelines["2. Top-80 Mutual Info Features + HGB"] = (
    selector.fit_transform(X_train, y_train),
    selector.transform(X_test),
)

# C. Hybrid: PCA-40 + LDA-10 (Linear orthogonal + class-separating directions)
pca40 = PCA(n_components=40, random_state=RANDOM_SEED)
lda10 = LDA(n_components=len(class_names) - 1)
pipelines["3. Hybrid PCA-40 + LDA-10 + HGB"] = (
    np.hstack([pca40.fit_transform(X_train), lda10.fit_transform(X_train, y_train)]),
    np.hstack([pca40.transform(X_test), lda10.transform(X_test)]),
)

# D. Full 328 Features (No dimensionality bottleneck)
pipelines["4. All 328 Features (No Reduction) + HGB"] = (X_train, X_test)

# Evaluate all four with identical HistGradientBoostingClassifier
rows = []
preds_store = {}
for name, (Z_tr, Z_te) in pipelines.items():
    clf = HistGradientBoostingClassifier(**hgb_cfg)
    clf.fit(Z_tr, y_train)
    preds = clf.predict(Z_te)
    preds_store[name] = preds
    rows.append({
        "Pipeline": name,
        "Test Accuracy (%)": round(accuracy_score(y_test, preds) * 100, 4),
        "Macro F1": round(f1_score(y_test, preds, average="macro"), 4),
        "Weighted F1": round(f1_score(y_test, preds, average="weighted"), 4),
    })

res_df = pd.DataFrame(rows).sort_values("Macro F1", ascending=False)
print("=" * 72)
print("COMPARISON: UMAP-40 vs. LINEAR/FEATURE-SELECTION/FULL-SPACE HGB")
print("=" * 72)
print(res_df.to_string(index=False))

winner_name = res_df.iloc[0]["Pipeline"]
print("\n" + "=" * 72)
print(f"WINNING PIPELINE CLASSIFICATION REPORT: {winner_name}")
print("=" * 72)
print(
    classification_report(
        y_test, preds_store[winner_name], target_names=class_names, digits=4
    )
)

COMPARISON: UMAP-40 vs. LINEAR/FEATURE-SELECTION/FULL-SPACE HGB
                                Pipeline  Test Accuracy (%)  Macro F1  Weighted F1
4. All 328 Features (No Reduction) + HGB            82.8221    0.7040       0.8237
    2. Top-80 Mutual Info Features + HGB            81.5951    0.6757       0.8121
         3. Hybrid PCA-40 + LDA-10 + HGB            77.9141    0.6608       0.7796
                        1. UMAP-40 + HGB            64.4172    0.4311       0.6225

WINNING PIPELINE CLASSIFICATION REPORT: 4. All 328 Features (No Reduction) + HGB
                   precision    recall  f1-score   support

          Cycling     0.8361    0.9273    0.8793        55
         Football     0.7927    0.8333    0.8125        78
          Jogging     0.5000    1.0000    0.6667         2
         JumpRope     0.8824    0.6522    0.7500        23
          Pushups     1.0000    0.3333    0.5000         3
          Sitting     0.9855    0.9784    0.9819       139
         Swimming     0.7